In [5]:
import sqlite3
import pandas as pd

# built into Python
# for readable output

con = sqlite3.connect(r'C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\data\raw\airtel.db') # two dots: notebooks/

def q(sql):
    return pd.read_sql(sql, con)

# the same helper as
# the IPL project

In [8]:
def run_sql_file(path):                         # one .sql file
    with open(path, 'r') as f:                  # read only
        script = f.read()                       # whole file
    con.executescript(script)                   # every statement
    con.commit()                                # save to the .db
    print('ran', path)                          # show what ran
    
run_sql_file(r'C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\sql\practice\00_test.sql')              # apply the file
q('SELECT COUNT(*) AS n FROM customers;')       # check it worke

ran C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\sql\practice\00_test.sql


,n
0,22000


In [10]:
run_sql_file(r'C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\sql\practice\01_revenue_customer.sql')

ran C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\sql\practice\01_revenue_customer.sql


In [125]:
q("""SELECT
    COUNT(*) AS rows_,
    SUM(recharge_count = 0) AS paid_zero,
    ROUND(AVG(actual_arpu_6m), 2) AS actual,
    (
        SELECT ROUND(AVG(arpu_inr), 2)
        FROM customers
    ) AS stated
FROM v_revenue_customer;""")

,rows_,paid_zero,actual,stated
0,22000,3711,344.46,421.65


In [126]:
q("""SELECT *
FROM v_revenue_customer
WHERE customer_id IN (4, 9);""")

,customer_id,recharge_count,recharge_total_6m,actual_arpu_6m
0,4,0,0,0.00
1,9,5,1745,290.83


In [11]:
run_sql_file(r'C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\sql\practice\02_usage_customer.sql')


ran C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\sql\practice\02_usage_customer.sql


In [12]:
q("""SELECT
    COUNT(*) AS total_customers,

    SUM(usage_days = 0) AS no_usage_customers,

    ROUND(AVG(mb_per_active_day), 2)
        AS avg_mb_per_active_day,

    ROUND(AVG(mb_per_observed_day), 2)
        AS avg_mb_per_observed_day

FROM v_usage_customer;""")


,total_customers,no_usage_customers,avg_mb_per_active_day,avg_mb_per_observed_day
0,22000,4,1404.44,524.05


In [13]:
q("""SELECT *
FROM v_usage_customer
WHERE customer_id IN (9, 266);""")

,customer_id,usage_days,total_data_mb,total_voice_min,total_sms,roaming_days,mb_per_active_day,mb_per_observed_day,last_usage_date,recency_days
0,9,65,100065.3,3292,43,3,1539.47,552.85,2025-06-28,2.0
1,266,0,0.0,0,0,0,NaN,0.00,None,NaN


In [14]:
run_sql_file(r'C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\sql\practice\03_complaint_customer.sql')

ran C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\sql\practice\03_complaint_customer.sql


In [131]:
q("""SELECT
    COUNT(*) AS total_customers,

    SUM(complaint_count = 0) AS never_complained,

    SUM(
        complaint_count > 0
        AND unresolved_count = 0
    ) AS all_resolved,

    SUM(unresolved_count > 0) AS has_open_complaints

FROM v_complaint_customer;""")


,total_customers,never_complained,all_resolved,has_open_complaints
0,22000,6631,8689,6680


In [132]:
q("""SELECT *
FROM v_complaint_customer
WHERE customer_id IN (9, 15);""")

,customer_id,complaint_count,unresolved_count,resolved_count,avg_resolution_days
0,9,3,0,3,9.33
1,15,1,1,0,NaN


In [15]:
run_sql_file(r'C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\sql\practice\04_plans_clean.sql')

ran C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\sql\practice\04_plans_clean.sql


In [16]:
q("""SELECT *
FROM v_plans_clean;""")

,plan_id,plan_name,plan_type,price_inr,validity_days,data_quota,quota_basis,quota_gb,monthly_gb,ott_benefit,is_5g
0,1,Prepaid Lite 99,Prepaid,99,14,0.5 GB/day,per day,0.5,15.0,None,0
1,2,Prepaid Smart 155,Prepaid,155,24,1 GB/day,per day,1.0,30.0,None,0
2,3,Prepaid Value 209,Prepaid,209,28,1 GB/day,per day,1.0,30.0,None,0
3,4,Prepaid Popular 299,Prepaid,299,28,1.5 GB/day,per day,1.5,45.0,None,1
4,5,Prepaid Data 349,Prepaid,349,28,2.5 GB/day,per day,2.5,75.0,None,1
5,6,Prepaid Entertainment 399,Prepaid,399,28,2.5 GB/day,per day,2.5,75.0,OTT-Lite,1
6,7,Prepaid Max 479,Prepaid,479,56,1.5 GB/day,per day,1.5,45.0,None,1
7,8,Prepaid Quarterly 719,Prepaid,719,84,2 GB/day,per day,2.0,60.0,None,1
8,9,Prepaid Annual 1799,Prepaid,1799,365,2 GB/day,per day,2.0,60.0,OTT,1
9,10,Postpaid 399,Postpaid,399,30,40 GB/month,per month,40.0,40.0,OTT-Lite,1


In [135]:
q("""SELECT
    plan_id,
    plan_type,
    data_quota,
    quota_basis,
    monthly_gb
FROM v_plans_clean
ORDER BY plan_id;""")

,plan_id,plan_type,data_quota,quota_basis,monthly_gb
0,1,Prepaid,0.5 GB/day,per day,15.0
1,2,Prepaid,1 GB/day,per day,30.0
2,3,Prepaid,1 GB/day,per day,30.0
3,4,Prepaid,1.5 GB/day,per day,45.0
4,5,Prepaid,2.5 GB/day,per day,75.0
5,6,Prepaid,2.5 GB/day,per day,75.0
6,7,Prepaid,1.5 GB/day,per day,45.0
7,8,Prepaid,2 GB/day,per day,60.0
8,9,Prepaid,2 GB/day,per day,60.0
9,10,Postpaid,40 GB/month,per month,40.0


In [17]:
import sqlite3 
conn = sqlite3.connect(r"C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\data\raw\airtel.db") 
with open(r"C:\Users\thway\OneDrive\Desktop\airtel-telecom-analytics\sql\practice\05_usage_month.sql", "r") as file:    
    sql_script = file.read() 
    conn.executescript(sql_script) 
    conn.commit() 
    print("v_usage_month created successfully!") 
    conn.close()

v_usage_month created successfully!


In [137]:
q("""SELECT COUNT(*) AS total_rows
FROM v_usage_month;""")

,total_rows
0,117832


In [ ]:
q("""SELECT
    COUNT(DISTINCT customer_id) AS distinct_subscribers,
    COUNT(DISTINCT month) AS distinct_months
FROM v_usage_month;""")1    

,distinct_subscribers,distinct_months
0,21996,6


In [139]:
q("""SELECT SUM(usage_days) AS total_usage_days
FROM v_usage_month;""")

,total_usage_days
0,1326739


In [140]:
q("""SELECT
    customer_id,
    month,
    usage_days,
    total_mb,
    total_voice,
    roaming_days
FROM v_usage_month
WHERE customer_id = 9
ORDER BY month;""")

,customer_id,month,usage_days,total_mb,total_voice,roaming_days
0,9,2025-01,11,11351.6,391,0
1,9,2025-02,7,10922.4,375,0
2,9,2025-03,9,13677.9,441,0
3,9,2025-04,9,10020.3,379,0
4,9,2025-05,18,34858.7,1075,1
5,9,2025-06,11,19234.4,631,2
